# 🎮 Breakout DQN — Entrenamiento Completo
**Universidad Nacional del Altiplano — Puno**  
**Escuela Profesional de Ingeniería de Sistemas**  
**Curso: ROBOTICA**

---

## 📋 Pipeline de Entrenamiento

| Etapa | Descripción |
|-------|-------------|
| 1. Entorno | Breakout simulado con 3 niveles |
| 2. Red DQN | Arquitectura `9→256→256→128→64→3` con Dropout |
| 3. Memoria | Experience Replay con buffer de 50 000 |
| 4. Estrategia | ε-greedy con decaimiento exponencial |
| 5. Target Net | Red objetivo actualizada cada 10 episodios |
| 6. Análisis | Gráficas de recompensa, score, loss y epsilon |
| 7. Exportación | Modelo `.pth` + `training_results.json` para Flask |

> ⚡ **Recomendado**: Activar GPU en `Runtime → Change runtime type → T4 GPU`

---
## 📦 Celda 1 — Instalación de dependencias

In [1]:
!pip install torch torchvision matplotlib numpy tqdm --quiet

import torch
import numpy as np
import matplotlib.pyplot as plt
import json, os, random, time
from collections import deque
from tqdm.notebook import tqdm
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ PyTorch {torch.__version__}')
print(f'📱 Dispositivo: {device}')
if torch.cuda.is_available():
    print(f'🚀 GPU: {torch.cuda.get_device_name(0)}')
    print(f'💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  Sin GPU — entrenamiento más lento en CPU')


✅ PyTorch 2.11.0+cu128
📱 Dispositivo: cuda
🚀 GPU: Tesla T4
💾 VRAM: 15.6 GB


---
## ⚙️ Celda 2 — Hiperparámetros configurables

In [2]:
CONFIG = {
    # Entrenamiento
    'TOTAL_EPISODES'    : 500,
    'MAX_STEPS'         : 2000,
    'WARMUP_EPISODES'   : 20,
    # DQN Core
    'GAMMA'             : 0.99,
    'LEARNING_RATE'     : 0.0005,
    'BATCH_SIZE'        : 64,
    'MEMORY_SIZE'       : 50000,
    'TARGET_UPDATE_FREQ': 10,
    # Epsilon
    'EPSILON_START'     : 1.0,
    'EPSILON_END'       : 0.01,
    'EPSILON_DECAY'     : 0.995,
    # Red
    'STATE_SIZE'        : 9,
    'ACTION_SIZE'       : 3,
    'DROPOUT_RATE'      : 0.2,
    # Curriculum
    'CURRICULUM'        : True,
    'CURRICULUM_THRESH' : 0.7,
    # Guardado
    'SAVE_EVERY'        : 100,
    'OUTPUT_MODEL'      : 'breakout_dqn_trained.pth',
    'OUTPUT_JSON'       : 'training_results.json',
}

print('📋 Configuración cargada:')
for k, v in CONFIG.items():
    print(f'   {k:25s}: {v}')


📋 Configuración cargada:
   TOTAL_EPISODES           : 500
   MAX_STEPS                : 2000
   WARMUP_EPISODES          : 20
   GAMMA                    : 0.99
   LEARNING_RATE            : 0.0005
   BATCH_SIZE               : 64
   MEMORY_SIZE              : 50000
   TARGET_UPDATE_FREQ       : 10
   EPSILON_START            : 1.0
   EPSILON_END              : 0.01
   EPSILON_DECAY            : 0.995
   STATE_SIZE               : 9
   ACTION_SIZE              : 3
   DROPOUT_RATE             : 0.2
   CURRICULUM               : True
   CURRICULUM_THRESH        : 0.7
   SAVE_EVERY               : 100
   OUTPUT_MODEL             : breakout_dqn_trained.pth
   OUTPUT_JSON              : training_results.json


---
## 🏗️ Celda 3 — Entorno Breakout con 3 niveles

In [3]:
LEVEL_CONFIG = {
    1: {'rows': 5, 'cols': 8, 'ball_speed': 6,  'paddle_width': 100,
        'paddle_speed': 16, 'brick_width': 78, 'brick_height': 24},
    2: {'rows': 6, 'cols': 8, 'ball_speed': 7,  'paddle_width': 85,
        'paddle_speed': 18, 'brick_width': 78, 'brick_height': 22},
    3: {'rows': 7, 'cols': 8, 'ball_speed': 8,  'paddle_width': 70,
        'paddle_speed': 20, 'brick_width': 78, 'brick_height': 20},
}

class BreakoutEnv:
    WIDTH  = 700
    HEIGHT = 600

    def __init__(self, level=1):
        cfg = LEVEL_CONFIG[max(1, min(3, level))]
        self.level        = level
        self.ball_speed   = cfg['ball_speed']
        self.paddle_width = cfg['paddle_width']
        self.paddle_speed = cfg['paddle_speed']
        self.brick_rows   = cfg['rows']
        self.brick_cols   = cfg['cols']
        self.brick_w      = cfg['brick_width']
        self.brick_h      = cfg['brick_height']
        self.brick_pad    = 6
        self.brick_top    = 60
        total_bw = self.brick_cols * (self.brick_w + self.brick_pad) - self.brick_pad
        self.brick_left   = (self.WIDTH - total_bw) // 2
        self.paddle_h     = 14
        self.ball_r       = 8
        self.reset()

    def reset(self):
        self.paddle_x = self.WIDTH  // 2 - self.paddle_width // 2
        self.paddle_y = self.HEIGHT - 40
        self.ball_x   = float(self.WIDTH  // 2)
        self.ball_y   = float(self.HEIGHT - 80)
        angle         = random.uniform(-0.5, 0.5)
        self.ball_dx  = self.ball_speed * angle
        self.ball_dy  = -self.ball_speed
        self.bricks   = self._make_bricks()
        self.score    = 0
        self.lives    = 3
        self.done     = False
        return self._state()

    def _make_bricks(self):
        bricks = []
        for r in range(self.brick_rows):
            for c in range(self.brick_cols):
                bricks.append({
                    'x': c * (self.brick_w + self.brick_pad) + self.brick_left,
                    'y': r * (self.brick_h + self.brick_pad) + self.brick_top,
                    'active': True, 'row': r
                })
        return bricks

    def _state(self):
        bricks_left  = sum(1 for b in self.bricks if b['active'])
        total_bricks = self.brick_rows * self.brick_cols
        active = [b for b in self.bricks if b['active']]
        if active:
            c = min(active, key=lambda b:
                abs(b['x'] + self.brick_w/2 - self.ball_x) +
                abs(b['y'] + self.brick_h/2 - self.ball_y))
            bdx = (c['x'] + self.brick_w/2 - self.ball_x) / self.WIDTH
            bdy = (c['y'] + self.brick_h/2 - self.ball_y) / self.HEIGHT
        else:
            bdx = bdy = 0.0
        paddle_center = self.paddle_x + self.paddle_width / 2
        return np.array([
            self.paddle_x / self.WIDTH,
            self.ball_x   / self.WIDTH,
            self.ball_y   / self.HEIGHT,
            self.ball_dx  / (2 * self.ball_speed),
            self.ball_dy  / (2 * self.ball_speed),
            bricks_left   / total_bricks,
            (paddle_center - self.ball_x) / self.WIDTH,
            bdx, bdy
        ], dtype=np.float32)

    def step(self, action):
        reward = 0.0
        if action == 0:
            self.paddle_x = max(0, self.paddle_x - self.paddle_speed)
        elif action == 2:
            self.paddle_x = min(self.WIDTH - self.paddle_width,
                                self.paddle_x + self.paddle_speed)
        self.ball_x += self.ball_dx
        self.ball_y += self.ball_dy
        if self.ball_x - self.ball_r <= 0:
            self.ball_dx = abs(self.ball_dx); self.ball_x = self.ball_r
        elif self.ball_x + self.ball_r >= self.WIDTH:
            self.ball_dx = -abs(self.ball_dx); self.ball_x = self.WIDTH - self.ball_r
        if self.ball_y - self.ball_r <= 0:
            self.ball_dy = abs(self.ball_dy); self.ball_y = self.ball_r
        hit_paddle = (
            self.ball_y + self.ball_r >= self.paddle_y and
            self.ball_y - self.ball_r <= self.paddle_y + self.paddle_h and
            self.ball_x >= self.paddle_x - self.ball_r and
            self.ball_x <= self.paddle_x + self.paddle_width + self.ball_r and
            self.ball_dy > 0
        )
        if hit_paddle:
            self.ball_dy = -abs(self.ball_dy)
            hit_pos = (self.ball_x - self.paddle_x) / self.paddle_width
            self.ball_dx = self.ball_speed * (hit_pos - 0.5) * 2.5
            self.ball_dx = float(np.clip(self.ball_dx,
                                         -self.ball_speed * 1.3,
                                          self.ball_speed * 1.3))
            self.ball_y = self.paddle_y - self.ball_r - 1
            reward += 1.0
        for brick in self.bricks:
            if not brick['active']:
                continue
            bx, by = brick['x'], brick['y']
            if (self.ball_x + self.ball_r >= bx and
                self.ball_x - self.ball_r <= bx + self.brick_w and
                self.ball_y + self.ball_r >= by and
                self.ball_y - self.ball_r <= by + self.brick_h):
                brick['active'] = False
                self.score += 10
                reward += 10.0
                ol  = (self.ball_x + self.ball_r) - bx
                or_ = (bx + self.brick_w) - (self.ball_x - self.ball_r)
                ot  = (self.ball_y + self.ball_r) - by
                ob  = (by + self.brick_h) - (self.ball_y - self.ball_r)
                if min(ot, ob) <= min(ol, or_):
                    self.ball_dy = -self.ball_dy
                else:
                    self.ball_dx = -self.ball_dx
                break
        if self.ball_y + self.ball_r >= self.HEIGHT:
            self.lives -= 1
            reward -= 50.0
            if self.lives <= 0:
                self.done = True; reward -= 50.0
            else:
                self.ball_x = float(self.WIDTH // 2)
                self.ball_y = float(self.HEIGHT - 80)
                angle = random.uniform(-0.5, 0.5)
                self.ball_dx = self.ball_speed * angle
                self.ball_dy = -self.ball_speed
        if all(not b['active'] for b in self.bricks):
            self.done = True; reward += 200.0
        reward += 0.05
        return self._state(), reward, self.done, {'score': self.score, 'lives': self.lives}

    def won(self):
        return all(not b['active'] for b in self.bricks)

env = BreakoutEnv(level=1)
s = env.reset()
print(f'Entorno OK — state shape: {s.shape}')
s2, r, done, info = env.step(2)
print(f'step() OK  — reward: {r:.2f}  done: {done}  score: {info["score"]}')


Entorno OK — state shape: (9,)
step() OK  — reward: 0.05  done: False  score: 0


---
## 🧠 Celda 4 — Red neuronal DQN y Agente

In [4]:
class DQNNetwork(nn.Module):
    def __init__(self, state_size, action_size, dropout=0.2):
        super().__init__()
        self.fc1  = nn.Linear(state_size, 256)
        self.fc2  = nn.Linear(256, 256)
        self.fc3  = nn.Linear(256, 128)
        self.fc4  = nn.Linear(128, 64)
        self.fc5  = nn.Linear(64, action_size)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        x = F.relu(self.fc1(x)); x = self.drop(x)
        x = F.relu(self.fc2(x)); x = self.drop(x)
        x = F.relu(self.fc3(x))
        x = F.relu(self.fc4(x))
        return self.fc5(x)


class DQNAgent:
    def __init__(self, cfg):
        self.cfg         = cfg
        self.state_size  = cfg['STATE_SIZE']
        self.action_size = cfg['ACTION_SIZE']
        self.gamma       = cfg['GAMMA']
        self.epsilon     = cfg['EPSILON_START']
        self.eps_min     = cfg['EPSILON_END']
        self.eps_decay   = cfg['EPSILON_DECAY']
        self.batch_size  = cfg['BATCH_SIZE']
        self.memory      = deque(maxlen=cfg['MEMORY_SIZE'])
        self.model  = DQNNetwork(self.state_size, self.action_size,
                                  cfg['DROPOUT_RATE']).to(device)
        self.target = DQNNetwork(self.state_size, self.action_size,
                                  cfg['DROPOUT_RATE']).to(device)
        self.sync_target()
        self.optimizer = optim.Adam(self.model.parameters(), lr=cfg['LEARNING_RATE'])
        self.scheduler = optim.lr_scheduler.StepLR(self.optimizer,
                                                    step_size=100, gamma=0.9)

    def sync_target(self):
        self.target.load_state_dict(self.model.state_dict())

    def remember(self, s, a, r, s2, done):
        self.memory.append((s, a, r, s2, done))

    def act(self, state, training=True):
        if training and random.random() < self.epsilon:
            return random.randrange(self.action_size)
        s = torch.FloatTensor(state).unsqueeze(0).to(device)
        self.model.eval()
        with torch.no_grad():
            q = self.model(s)
        self.model.train()
        return q.argmax().item()

    def learn(self):
        if len(self.memory) < self.batch_size:
            return 0.0
        batch = random.sample(self.memory, self.batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        states      = torch.FloatTensor(np.array(states)).to(device)
        actions     = torch.LongTensor(actions).unsqueeze(1).to(device)
        rewards     = torch.FloatTensor(rewards).to(device)
        next_states = torch.FloatTensor(np.array(next_states)).to(device)
        dones       = torch.FloatTensor(dones).to(device)
        q_current = self.model(states).gather(1, actions).squeeze()
        with torch.no_grad():
            next_actions = self.model(next_states).argmax(1, keepdim=True)
            q_next   = self.target(next_states).gather(1, next_actions).squeeze()
            q_target = rewards + (1.0 - dones) * self.gamma * q_next
        loss = F.smooth_l1_loss(q_current, q_target)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=10.0)
        self.optimizer.step()
        return loss.item()

    def decay_epsilon(self):
        self.epsilon = max(self.eps_min, self.epsilon * self.eps_decay)

    def save(self, path):
        torch.save(self.model.state_dict(), path)
        print(f'Modelo guardado -> {path}')

    def load(self, path):
        self.model.load_state_dict(torch.load(path, map_location=device))
        self.sync_target()
        print(f'Modelo cargado  <- {path}')


agent_test = DQNAgent(CONFIG)
x_test = torch.randn(1, CONFIG['STATE_SIZE']).to(device)
q_test = agent_test.model(x_test)
print(f'Red DQN OK — Q-values shape: {q_test.shape}')
params = sum(p.numel() for p in agent_test.model.parameters())
print(f'Parametros totales: {params:,}')
del agent_test, x_test, q_test


Red DQN OK — Q-values shape: torch.Size([1, 3])
Parametros totales: 109,699


---
## 🚀 Celda 5 — Loop de entrenamiento principal

In [5]:
def train(cfg):
    agent   = DQNAgent(cfg)
    history = {'rewards': [], 'scores': [], 'losses': [],
               'epsilons': [], 'levels': [], 'victories': []}
    current_level = 1
    win_window    = deque(maxlen=20)
    best_score    = 0

    print('=' * 60)
    print('ENTRENAMIENTO BREAKOUT DQN')
    print(f'Episodios  : {cfg["TOTAL_EPISODES"]}')
    print(f'Dispositivo: {device}')
    print(f'Curriculum : {cfg["CURRICULUM"]}')
    print('=' * 60)

    pbar = tqdm(range(1, cfg['TOTAL_EPISODES'] + 1), desc='Entrenando')
    for ep in pbar:
        env   = BreakoutEnv(level=current_level)
        state = env.reset()
        total_reward = 0.0
        ep_losses    = []
        training_flag = (ep > cfg['WARMUP_EPISODES'])

        for step in range(cfg['MAX_STEPS']):
            action = agent.act(state, training=training_flag)
            next_state, reward, done, info = env.step(action)
            agent.remember(state, action, reward, next_state, done)
            loss = agent.learn()
            if loss > 0:
                ep_losses.append(loss)
            total_reward += reward
            state = next_state
            if done:
                break

        agent.decay_epsilon()

        if ep % cfg['TARGET_UPDATE_FREQ'] == 0:
            agent.sync_target()

        if ep % 100 == 0:
            agent.scheduler.step()

        won = env.won()
        win_window.append(1 if won else 0)
        win_rate = float(np.mean(win_window)) if win_window else 0.0

        history['rewards'].append(float(total_reward))
        history['scores'].append(int(info['score']))
        history['losses'].append(float(np.mean(ep_losses)) if ep_losses else 0.0)
        history['epsilons'].append(float(agent.epsilon))
        history['levels'].append(current_level)
        history['victories'].append(int(won))

        if (cfg['CURRICULUM'] and len(win_window) == 20
                and win_rate >= cfg['CURRICULUM_THRESH']
                and current_level < 3):
            current_level += 1
            win_window.clear()
            print(f'\nAvanzando al Nivel {current_level}! (win rate: {win_rate:.0%})')

        if info['score'] > best_score:
            best_score = info['score']
            agent.save('breakout_dqn_best.pth')

        if ep % cfg['SAVE_EVERY'] == 0:
            agent.save(f'ckpt_ep{ep}.pth')

        pbar.set_postfix({
            'score': info['score'],
            'eps'  : f"{agent.epsilon:.3f}",
            'lvl'  : current_level,
            'win%' : f"{win_rate:.0%}",
            'loss' : f"{history['losses'][-1]:.4f}"
        })

    agent.save(cfg['OUTPUT_MODEL'])
    with open(cfg['OUTPUT_JSON'], 'w') as f:
        json.dump(history, f)
    print(f'Historial guardado -> {cfg["OUTPUT_JSON"]}')
    return agent, history


# INICIAR ENTRENAMIENTO
start = time.time()
agent, history = train(CONFIG)
elapsed = time.time() - start
print(f'Tiempo total: {elapsed/60:.1f} min')
print(f'Mejor score: {max(history["scores"])}')
print(f'Victorias: {sum(history["victories"])}/{CONFIG["TOTAL_EPISODES"]}')


ENTRENAMIENTO BREAKOUT DQN
Episodios  : 500
Dispositivo: cuda
Curriculum : True


Entrenando:   0%|          | 0/500 [00:00<?, ?it/s]

Modelo guardado -> breakout_dqn_best.pth
Modelo guardado -> breakout_dqn_best.pth
Modelo guardado -> breakout_dqn_best.pth
Modelo guardado -> breakout_dqn_best.pth
Modelo guardado -> ckpt_ep100.pth
Modelo guardado -> breakout_dqn_best.pth
Modelo guardado -> ckpt_ep200.pth
Modelo guardado -> ckpt_ep300.pth
Modelo guardado -> ckpt_ep400.pth
Modelo guardado -> ckpt_ep500.pth
Modelo guardado -> breakout_dqn_trained.pth
Historial guardado -> training_results.json
Tiempo total: 15.9 min
Mejor score: 240
Victorias: 0/500
